In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import date
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, inspect


In [2]:
load_dotenv()

True

In [3]:
db_url = os.getenv("DATABASE_URL")

In [4]:
engine = create_engine(db_url)

In [5]:
inspector = inspect(engine)

In [6]:
table_names = inspector.get_table_names()

for table in table_names:
    print(table)

sources
items
vulnerabilities
hn_seen_ids


In [ ]:
today = Path(f"../../../data/{date.today()}")
print(f"Loading from: {today}")

Loading from: ../../../data/2026-07-03


In [8]:
with engine.connect() as conn:
    total = pd.read_sql("SELECT COUNT(*) AS n FROM items", conn).iloc[0, 0]
    with_content = pd.read_sql("SELECT COUNT(*) AS n FROM items WHERE content IS NOT NULL", conn).iloc[0, 0]
    with_summary = pd.read_sql("SELECT COUNT(*) AS n FROM items WHERE summary IS NOT NULL", conn).iloc[0, 0]

print(f"Items:              {total}")
print(f"With content:       {with_content}")
print(f"With summary:       {with_summary}")

Items:              335
With content:       177
With summary:       335


In [ ]:
with engine.connect() as conn:
    df = pd.read_sql("""
        SELECT
            CASE
                WHEN summary IS NULL AND content IS NULL THEN 'no summary, no content'
                WHEN summary IS NULL AND content IS NOT NULL THEN 'no summary, content'
                WHEN summary IS NOT NULL AND content IS NULL THEN 'summary, no content'
                WHEN summary IS NOT NULL AND content IS NOT NULL THEN 'summary, content'
            END AS category,
            COUNT(*) AS count
        FROM items
        GROUP BY category
    """, conn)

df

In [ ]:
with engine.connect() as conn:
    titles = pd.read_sql("SELECT title FROM items WHERE summary IS NOT NULL", conn)

for t in titles['title']:
    print(t)